# 02. 상품링크 기반 공급사 · 사업자정보 크롤링

`01_crawl_product_list.ipynb`가 만든 상품링크를 입력으로 받아,
1) 상품 상세페이지에서 공급사 연락처(전화/휴대폰/이메일/주소 등)와 **발주링크**를 추출하고,
2) 같은 세션으로 발주링크에 접속해 **사업자번호 · 대표자 · 업태 · 종목** 등 공급사 사업자정보를 추가로 추출합니다.

- **입력**: `partner_goods_full.xlsx` (01 노트북의 출력)
- **출력**: `supplier_detail_result.xlsx`
- **필요 환경변수**: `.env`의 `GIFTCO_SUPPLIER_COOKIE` (giftco.co.kr 로그인 세션 쿠키, 01과 동일한 값 사용)
- **실행 방법**: 셀을 순서대로 실행하면 마지막 셀에서 크롤링이 시작됩니다. 업체별로 **가장 최근 등록 상품 1건**만 대상으로 하며, 발주페이지 크롤링이 필요 없다면 `CRAWL_ORDER_PAGE = False`로 설정해 상세페이지 정보만 수집할 수도 있습니다.


In [ ]:
import os
import re
import time
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from tqdm import tqdm

load_dotenv()

# ============================================================
# 0. 설정  ──  본인 환경에 맞게 수정
# ============================================================
GIFTCO_SUPPLIER_COOKIE = os.environ.get("GIFTCO_SUPPLIER_COOKIE", "").strip()
if not GIFTCO_SUPPLIER_COOKIE:
    raise RuntimeError(
        ".env에 GIFTCO_SUPPLIER_COOKIE가 설정되지 않았습니다. "
        "giftco.co.kr 로그인 후 F12 > Network 탭에서 아무 요청이나 클릭 > "
        "Request Headers의 Cookie 값을 통째로 복사해 .env에 넣어주세요."
    )

DATA_DIR = "../data"
EXCEL_IN = f"{DATA_DIR}/partner_goods_full.xlsx"          # 입력: 01 노트북의 출력
EXCEL_OUT = f"{DATA_DIR}/supplier_detail_result.xlsx"        # 출력

TIMEOUT = 15
SLEEP_SEC = 0.5

CRAWL_ORDER_PAGE = True   # 발주페이지(사업자번호 등)까지 크롤링할지 여부

HEADERS = {
    "Cookie": GIFTCO_SUPPLIER_COOKIE,
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"
    ),
}

# 상세페이지 공급사 표에서 뽑을 항목 (화면 기준)
WANT_FIELDS = [
    "전화번호", "휴대폰번호", "팩스", "이메일", "주소",
    "상품등록일", "상품최근수정일",
    "인쇄가능여부", "한박스당수량", "한박스당배송비", "이미지사용",
    "발주링크", "gs_id",
]

# 표의 th 텍스트(공백 제거) → 위 필드명 매핑
WANT_NORM = {
    "전화번호": "전화번호",
    "휴대폰번호": "휴대폰번호",
    "팩스": "팩스",
    "이메일": "이메일",
    "주소": "주소",
    "상품등록일": "상품등록일",
    "상품최근수정일": "상품최근수정일",
    "인쇄가능여부": "인쇄가능여부",
    "한박스당수량": "한박스당수량",
    "한박스당배송비": "한박스당배송비",
    "이미지사용": "이미지사용",
}

# 발주페이지에서 뽑을 '공급처' 항목 (표 오른쪽 칸)
SUPPLIER_FIELDS = ["상호", "사업자번호", "대표자", "업태", "종목", "전화", "팩스", "이메일", "주소"]
SUPPLIER_NORM = {k: k for k in SUPPLIER_FIELDS}   # 라벨에 공백 없음


In [ ]:
# ============================================================
# 1. 공통 유틸 + 파서
# ============================================================
def _norm(s):
    """공백/줄바꿈 모두 제거해서 비교용 키로."""
    return re.sub(r"\s+", "", s or "")


def clean_text(node):
    """td 안의 텍스트만 깔끔하게. 버튼/공백/줄바꿈 정리."""
    txt = node.get_text(" ", strip=True)
    txt = re.sub(r"\s*(공급사)?발주하기\s*", " ", txt)   # 버튼 라벨 제거
    txt = re.sub(r"\s+", " ", txt).strip()
    return txt


def _cell_value(td):
    """공급처 td 값 추출. 보이는 텍스트 우선, 없으면 hidden input value."""
    txt = re.sub(r"\s+", " ", td.get_text(" ", strip=True)).strip()
    if txt:
        return txt
    inp = td.find("input")
    if inp and inp.get("value"):
        return inp["value"].strip()
    return ""


def parse_supplier_table(html, base_url=""):
    """상세페이지에서 공급사 표(연락처)와 발주링크를 추출."""
    soup = BeautifulSoup(html, "html.parser")

    # class에 'wfull' 포함 + '공급사' th 가 있는 표 찾기
    target_table = None
    for table in soup.find_all("table", class_=lambda c: c and "wfull" in c):
        ths = [th.get_text(strip=True) for th in table.find_all("th")]
        if "공급사" in ths:
            target_table = table
            break

    result = {f: "" for f in WANT_FIELDS}
    if target_table is None:
        return result, False  # 표 못 찾음(로그인 필요/구조 변경 가능)

    for tr in target_table.find_all("tr"):
        th = tr.find("th")
        td = tr.find("td")
        if not th or not td:
            continue

        th_text = th.get_text(strip=True)

        # 일반 항목 매핑
        key = WANT_NORM.get(_norm(th_text))
        if key:
            result[key] = clean_text(td)

        # 공급사 행에서 '공급사발주하기' 링크 추출 → 절대경로로
        if th_text == "공급사":
            a = td.find("a", href=True)
            if a:
                href = a["href"]
                result["발주링크"] = urljoin(base_url, href) if base_url else href
                m = re.search(r"gs_id=(\d+)", href)
                if m:
                    result["gs_id"] = m.group(1)

    return result, True


def parse_order_page(html):
    """발주페이지에서 '공급처'(오른쪽 칸)의 사업자번호/대표자/업태/종목 등을 추출.
    각 행 구조: th(라벨) td(가맹점값) th(라벨) td(공급처값) → 두 번째 th/td 쌍이 공급처.
    """
    soup = BeautifulSoup(html, "html.parser")
    data = {}

    container = soup.find("div", class_=lambda c: c and "tbl_frm03" in c)
    scope = container if container else soup
    table = scope.find("table", class_=lambda c: c and "wfull" in c)
    if table is None:
        return data

    for tr in table.find_all("tr"):
        cells = tr.find_all(["th", "td"], recursive=False)
        # th td th td  → 오른쪽(공급처) 쌍은 인덱스 2(th), 3(td)
        if len(cells) >= 4 and cells[2].name == "th" and cells[3].name == "td":
            label = _norm(cells[2].get_text(strip=True))
            key = SUPPLIER_NORM.get(label)
            if key:
                data[f"공급처_{key}"] = _cell_value(cells[3])
    return data


In [ ]:
# ============================================================
# 2. 엑셀 → 업체명별 '최근 상품' 링크 만들기
# ============================================================
def build_targets(excel_path):
    df = pd.read_excel(excel_path, dtype=str)
    df["번호_int"] = df["번호"].astype(int)

    # 업체명별 상품 수
    counts = df.groupby("업체명").size().rename("상품수")

    # 업체명별로 '번호'가 가장 큰(=가장 최근) 행만
    idx = df.groupby("업체명")["번호_int"].idxmax()
    latest = df.loc[idx].sort_values("번호_int", ascending=False)
    targets = latest[["업체명", "업체코드", "번호", "상품링크"]].reset_index(drop=True)

    # 상품수 병합
    targets = targets.merge(counts, on="업체명", how="left")

    print(f"[info] 대상 업체 수: {len(targets)}")
    return targets


In [ ]:
# ============================================================
# 3. 크롤링 실행
#    4-1. 상세페이지에서 공급사 연락처 + 발주링크 추출
#    4-2. (CRAWL_ORDER_PAGE=True인 경우) 같은 세션으로 발주링크 접속 → 사업자정보 추출
# ============================================================
def crawl():
    targets = build_targets(EXCEL_IN)

    session = requests.Session()
    session.headers.update(HEADERS)

    rows = []
    for i, r in tqdm(targets.iterrows(), total=len(targets)):
        url = r["상품링크"]
        rec = {
            "업체명": r["업체명"],
            "업체코드": r["업체코드"],
            "상품수": r["상품수"],
            "상품링크": url,
        }

        # --- 4-1. 상세페이지 ---
        try:
            resp = session.get(url, timeout=TIMEOUT)
            resp.encoding = resp.apparent_encoding or resp.encoding  # EUC-KR/UTF-8 대비
            data, found = parse_supplier_table(resp.text, base_url=url)
            rec.update(data)
            rec["상태"] = "OK" if found else "표없음(로그인필요?)"
        except Exception as e:
            for f in WANT_FIELDS:
                rec[f] = ""
            rec["상태"] = f"에러: {e}"

        # --- 4-2. 발주페이지 (같은 session으로 쿠키 유지) ---
        if CRAWL_ORDER_PAGE and rec.get("발주링크"):
            try:
                time.sleep(SLEEP_SEC)
                r2 = session.get(rec["발주링크"], timeout=TIMEOUT)
                r2.encoding = r2.apparent_encoding or r2.encoding
                rec.update(parse_order_page(r2.text))
                rec["발주_상태"] = "OK"
            except Exception as e:
                rec["발주_상태"] = f"에러: {e}"

        rows.append(rec)
        # if len(rows) % 50 == 0:
        #     print(f"[{i+1}/{len(targets)}] {rec['업체명']:<20} | "
        #           f"{rec.get('휴대폰번호', ''):<15} | {rec['상태']}")
        time.sleep(SLEEP_SEC)

    # --- 4-3. 저장 (고정 열 먼저, 나머지 동적 열은 뒤에 자동 추가) ---
    supplier_cols = [f"공급처_{f}" for f in SUPPLIER_FIELDS]
    fixed = (["업체명", "업체코드", "상품수"] + WANT_FIELDS
             + supplier_cols + ["상품링크", "상태", "발주_상태"])

    df_out = pd.DataFrame(rows)
    ordered = [c for c in fixed if c in df_out.columns]
    extra = [c for c in df_out.columns if c not in ordered]
    df_out = df_out[ordered + extra]

    df_out.to_excel(EXCEL_OUT, index=False)
    print(f"\n[done] 저장 완료 → {EXCEL_OUT}  ({len(df_out)}개 업체)")
    return df_out


df_out = crawl()


## 4. 상품목록 + 공급사정보 조인 (상품조회툴 입력 파일 생성)

`supplier_product_viewer`(상품조회툴)는 상품 전체 목록에 공급사 정보가 합쳐진 파일을 사용합니다.
01의 출력(`partner_goods_full.xlsx`, 상품 전체)과 위 `crawl()`의 출력(`supplier_detail_result.xlsx`, 업체별 1행)을
**업체명 기준 left join**으로 합쳐 `products_with_supplier_info.xlsx`를 만듭니다.


In [ ]:
JOIN_OUTPUT_PATH = f"{DATA_DIR}/products_with_supplier_info.xlsx"


def build_full_join():
    goods = pd.read_excel(EXCEL_IN, dtype=str)      # partner_goods_full.xlsx: 상품 전체 (01 출력)
    supplier = pd.read_excel(EXCEL_OUT, dtype=str)  # supplier_detail_result.xlsx: 업체별 1행 (위 crawl() 출력)

    # 공급사 파일에서 상품 파일과 겹치지 않는 컬럼만 골라 '업체명' 기준으로 왼쪽 조인
    key = "업체명"
    supplier_extra_cols = [key] + [c for c in supplier.columns if c not in goods.columns]
    joined = goods.merge(supplier[supplier_extra_cols], on=key, how="left", indicator=True)

    matched = int((joined["_merge"] == "both").sum())
    joined = joined.drop(columns="_merge")

    joined.to_excel(JOIN_OUTPUT_PATH, index=False)
    print(f"[done] 조인 완료 → {JOIN_OUTPUT_PATH}  ({len(joined)}행, 공급사 정보 매칭 {matched}건)")
    return joined


df_join = build_full_join()
